# Time-doubling law for the scrambling of edge information in a nonintegrable Ising chain
**Reproduction notebook**: every figure (Figs. 2–8), Table I and the numbers quoted in the text.

Model (open chain, sites $j=0,\dots,L-1$, $J=1$):
$$H=-J\sum_j Z_jZ_{j+1}+K\sum_j Z_jZ_{j+2}-g\sum_j X_j ,\qquad K=-0.3 .$$

**How to use**
1. Set the switches in the *Settings* cell. Then run all cells from top to bottom.
2. Every computation writes its result to `data/*.npz`, and a result that is already there is loaded instead of recomputed. You can stop the notebook and restart it at any time.
3. Figures are written to `figs/` as PDF and PNG. Table I and the quoted numbers are printed at the end and saved in `figs/stats.json`.

**Cost on a laptop** (single node, 8–16 GB RAM)

| Part | What | Time |
|---|---|---|
| $L\le 12$ exact diagonalization (always run) | Figs. 2–8 at $L\le12$ | about 30–60 min |
| `RUN_L14 = True` | parity-sector ED at $L=14$ (Figs. 2, 7, 8, Table I) | about 1–2 h, about 6 GB RAM |
| `RUN_LARGE = True` | Chebyshev typicality at $L=16,18$ (Figs. 6b, 7a,b, Table I) | many hours (the $L=18$ runs dominate) |

A panel whose data are missing is drawn without those curves, and the notebook prints a notice.

Requirements: `numpy`, `scipy`, `matplotlib` (no other packages).

In [1]:
# ============================ Settings ============================
RUN_L14   = True    # parity-sector exact diagonalization at L = 14
RUN_LARGE = False   # sparse Chebyshev + typicality at L = 16 and 18 (slow)
DATA, FIGS = "data", "figs"

J, K0 = 1.0, -0.3                                   # couplings used throughout
import os, glob, json, time, numpy as np, scipy.sparse as sps
from scipy.special import jv
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import LogLocator, NullFormatter
os.makedirs(DATA, exist_ok=True); os.makedirs(FIGS, exist_ok=True)
T_SEC = np.logspace(-1, 9, 201)        # time grid for exact memories
WC    = np.logspace(-4, 0.3, 44)       # frequency windows for q(omega_c)
G_TAU = [0.25, 0.28, 0.32, 0.36, 0.40, 0.45, 0.50]   # fields of Table I
def fpath(n): return f"{DATA}/{n}.npz"
def have(n): return os.path.exists(fpath(n))
def load(n): return np.load(fpath(n))

## 1. Core engines
* **Full ED** (`H_chain`, `Spectrum`): dense $2^L\times 2^L$ Hamiltonian, used for OTOCs at $L\le12$.
* **Parity-sector ED** (`Sectors`): the parity $P=\prod_j X_j$ is conserved and $Z_j$ maps the $P=+1$ sector onto $P=-1$. Memories $A(t)=(Z_j|Z_j(t))$ and the zero-mode weight $q(\omega_c)$ follow exactly from the two sector spectra ($L\le14$).
* **Chebyshev typicality** (`Chain`): sparse Hamiltonian, Chebyshev time evolution of Haar-random states ($L=16,18$).

In [2]:
# ---------- basis helpers and the full Hamiltonian ----------
def zdiag(L, i):
    """Diagonal of Z_i in the computational basis (site 0 = most significant bit)."""
    idx = np.arange(2**L); return 1.0 - 2.0 * ((idx >> (L - 1 - i)) & 1)

def H_chain(L, g, K=K0, J=J):
    """Dense H = -J sum Z_i Z_{i+1} + K sum Z_i Z_{i+2} - g sum X_i (open boundaries)."""
    D = 2**L; idx = np.arange(D); H = np.zeros((D, D))
    d = -J * sum(zdiag(L, b) * zdiag(L, b + 1) for b in range(L - 1))
    d = d + K * sum(zdiag(L, b) * zdiag(L, b + 2) for b in range(L - 2))
    H[idx, idx] = d
    for i in range(L):
        H[idx ^ (1 << (L - 1 - i)), idx] += -g
    return H

class Spectrum:
    def __init__(self, H): self.E, self.P = np.linalg.eigh(H); self.D = len(self.E)
    def op(self, diag): return self.P.T @ (diag[:, None] * self.P)     # diagonal operator in the eigenbasis

def weights(sp, beta):
    if beta == 0: return np.full(sp.D, 1 / sp.D)
    p = np.exp(-beta * (sp.E - (sp.E.min() if beta > 0 else sp.E.max()))); return p / p.sum()

def autocorr(sp, M, times, beta=0.0):
    """A(t) = sum_mn p_m |M_mn|^2 e^{i(E_m-E_n)t}, exact."""
    U = np.exp(-1j * np.outer(sp.E - sp.E.mean(), np.asarray(times))); p = weights(sp, beta)
    return np.real(np.sum((p[:, None] * np.conj(U)) * ((M**2) @ U), axis=0))

def otoc_exact(sp, L, w_site, v_sites, times, beta=0.0):
    """C = 1 - Re Tr[rho W(t) V W(t) V] with W = Z_w, V = Z_v (exact trace)."""
    p = weights(sp, beta); M = sp.op(zdiag(L, w_site)); Vs = [sp.op(zdiag(L, i)) for i in v_sites]
    w = sp.E[:, None] - sp.E[None, :]; C = np.zeros((len(v_sites), len(times)))
    for k, t in enumerate(times):
        Wt = M * np.exp(1j * w * t)
        for a, V in enumerate(Vs):
            X = Wt @ V; C[a, k] = 1 - np.real(np.sum(p[:, None] * X * X.T))
    return C

def otoc_typical(sp, L, w_site, times, nstates=2, seed=0):
    """Infinite-temperature OTOC from Z_w to every Z_j, by dynamical typicality (Haar-random states)."""
    rng = np.random.default_rng(seed); M = sp.op(zdiag(L, w_site)); P = sp.P
    zs = np.array([zdiag(L, i) for i in range(L)]).T; C = np.zeros((L, len(times)))
    for s in range(nstates):
        psi = rng.normal(size=sp.D) + 1j * rng.normal(size=sp.D); psi /= np.linalg.norm(psi)
        Phi = np.concatenate([psi[:, None], P.T @ (zs * (P @ psi)[:, None])], axis=1)
        for k, t in enumerate(times):
            ph = np.exp(-1j * sp.E * t)
            WPhi = np.conj(ph)[:, None] * (M @ (ph[:, None] * Phi))
            Vchi = P.T @ (zs * (P @ WPhi[:, 0])[:, None])
            C[:, k] += 1 - np.real(np.sum(np.conj(Vchi) * WPhi[:, 1:], axis=0))
    return C / nstates

def crossing(ts, y, th):
    """First time at which y(t) >= th, interpolated on a log-time axis."""
    k = np.argmax(y >= th)
    if y[k] < th: return np.nan
    if k == 0: return ts[0]
    lt = np.log(ts); return float(np.exp(lt[k-1] + (th - y[k-1]) * (lt[k] - lt[k-1]) / (y[k] - y[k-1])))

In [3]:
# ---------- parity-sector exact diagonalization ----------
def zbits(L, s, j): return 1.0 - 2.0 * ((s >> (L - 1 - j)) & 1)

def sector_H(L, g, K=K0, J=J):
    """H in the P=+1 and P=-1 sectors. Basis |s,+-> = (|s> +- |s_bar>)/sqrt2 with s < 2^(L-1)."""
    D2 = 2 ** (L - 1); s = np.arange(D2)
    d = -J * sum(zbits(L, s, b) * zbits(L, s, b + 1) for b in range(L - 1))
    d = d + K * sum(zbits(L, s, b) * zbits(L, s, b + 2) for b in range(L - 2))
    Hp = np.diag(d.astype(float)); Hm = Hp.copy()
    for i in range(1, L):                              # flips that keep the top bit 0
        t = s ^ (1 << (L - 1 - i)); Hp[t, s] += -g; Hm[t, s] += -g
    r = s ^ (D2 - 1)                                   # flip of site 0: X_0|s,+-> = +-|r,+->
    Hp[r, s] += -g; Hm[r, s] += +g
    return Hp, Hm, s

class Sectors:
    def __init__(self, L, g, K=K0):
        self.L, self.g, self.K, self.D = L, g, K, 2 ** L
        Hp, Hm, self.s = sector_H(L, g, K)
        self.Ep, self.Vp = np.linalg.eigh(Hp); del Hp
        self.Em, self.Vm = np.linalg.eigh(Hm); del Hm
        self.E0 = 0.5 * (self.Ep.mean() + self.Em.mean())
    def Zpm(self, j):
        """<m,+|Z_j|n,-> in the eigenbases."""
        return self.Vp.T @ (zbits(self.L, self.s, j)[:, None] * self.Vm)
    def memory(self, M, times, chunk=40):
        """A(t) = (2/D) sum_mn M_mn^2 cos((E_m-E_n)t), exact for all t."""
        W = M ** 2; times = np.asarray(times); out = np.zeros(len(times))
        for a in range(0, len(times), chunk):
            tt = times[a:a + chunk]
            Up = np.exp(-1j * np.outer(self.Ep - self.E0, tt)); Um = np.exp(-1j * np.outer(self.Em - self.E0, tt))
            out[a:a + chunk] = np.real(np.sum(np.conj(Up) * (W @ Um), axis=0))
        return 2.0 * out / self.D
    def lowfreq_weight(self, M, wc_list):
        """q(w_c) = (1/D) sum_{|E_m-E_n|<w_c} |<m|Z|n>|^2  (weight of the slow / zero-mode part)."""
        W = (M ** 2) * (2.0 / self.D); out = np.zeros(len(wc_list))
        for a in range(0, len(self.Ep), 256):
            om = np.abs(self.Ep[a:a + 256, None] - self.Em[None, :])
            for k, wc in enumerate(wc_list): out[k] += W[a:a + 256][om < wc].sum()
        return out

def q_of(d, wc=0.1):  return float(np.interp(np.log(wc), np.log(d["wc"]), d["q0"]))
def tau_edge(t, A, q): return crossing(t, 1 - A / q, 0.5)        # A(tau_edge) = q/2

In [4]:
# ---------- sparse Chebyshev propagation with dynamical typicality (L = 16, 18) ----------
class Chain:
    def __init__(self, L, g, K=K0, J=J):
        self.L, self.g, self.D = L, g, 2**L; D = self.D; idx = np.arange(D)
        d = -J * sum(zdiag(L, b) * zdiag(L, b + 1) for b in range(L - 1)) + K * sum(zdiag(L, b) * zdiag(L, b + 2) for b in range(L - 2))
        rows = np.concatenate([idx] + [idx ^ (1 << (L - 1 - i)) for i in range(L)])
        cols = np.concatenate([idx] * (L + 1)); vals = np.concatenate([d] + [np.full(D, -g)] * L)
        self.a = 1.01 * (abs(J) * (L - 1) + abs(K) * (L - 2) + abs(g) * L)       # spectral bound
        self.H = sps.csr_matrix((vals / self.a, (rows, cols)), shape=(D, D))
    def evolve(self, v, t, max_step=40.0):
        if t == 0: return v
        nstep = int(np.ceil(self.a * abs(t) / max_step)); dt = t / nstep; x = self.a * abs(dt)
        N = int(x + 12 * x**(1 / 3) + 20); k = np.arange(N + 1)
        c = jv(k, x) * (-1j * np.sign(dt))**k; c[1:] *= 2
        for _ in range(nstep):
            T0 = v; T1 = self.H @ v; acc = c[0] * T0 + c[1] * T1
            for kk in range(2, N + 1):
                T2 = 2 * (self.H @ T1) - T0; acc += c[kk] * T2; T0, T1 = T1, T2
            v = acc
        return v

def random_state(D, rng):
    psi = rng.normal(size=D) + 1j * rng.normal(size=D); return psi / np.linalg.norm(psi)

def cheb_memory(ch, site, times, seed=0):
    rng = np.random.default_rng(seed); z = zdiag(ch.L, site); out = np.zeros(len(times))
    psi = random_state(ch.D, rng); v = np.stack([psi, z * psi], axis=1); tp = 0.0
    for k, t in enumerate(times):
        v = ch.evolve(v, t - tp); tp = t; out[k] = np.real(np.vdot(v[:, 0], z * v[:, 1]))
    return out

def cheb_otoc(ch, w_site, probes, times, seed=0):
    rng = np.random.default_rng(seed); zw = zdiag(ch.L, w_site)
    zs = np.stack([zdiag(ch.L, j) for j in probes], axis=1); C = np.zeros((len(probes), len(times)))
    psi = random_state(ch.D, rng); fwd = np.concatenate([psi[:, None], zs * psi[:, None]], axis=1); tp = 0.0
    for k, t in enumerate(times):
        fwd = ch.evolve(fwd, t - tp); tp = t
        back = ch.evolve(zw[:, None] * fwd, -t)
        F = np.sum(np.conj(zs * back[:, :1]) * back[:, 1:], axis=0); C[:, k] = 1 - F.real
    return C

def tgrid(tmax, per_decade=20):
    return np.logspace(-1, np.log10(tmax), int(per_decade * (np.log10(tmax) + 1)) + 1)

## 2. Checks of the engines
These checks take a few seconds. Expected output: the builder error and the sector error are about $10^{-15}$; the mapping check prints `True`; the typicality error is a few $10^{-2}$; the $K=0$ zero-mode weight matches $1-g^2$ to about $10^{-3}$.

In [5]:
_X = np.array([[0., 1.], [1., 0.]]); _Z = np.diag([1., -1.])
def _op(o, i, L): return np.kron(np.kron(np.eye(2**i), o), np.eye(2**(L - 1 - i)))
L_ = 6
Hk = sum(-J * _op(_Z, b, L_) @ _op(_Z, b + 1, L_) for b in range(L_ - 1)) \
   + sum(K0 * _op(_Z, b, L_) @ _op(_Z, b + 2, L_) for b in range(L_ - 2)) + sum(-0.4 * _op(_X, i, L_) for i in range(L_))
print("Hamiltonian builder error           :", np.abs(H_chain(L_, 0.4) - Hk).max())
# exact map U H(g,K) U^dag = -H(g,-K), U = prod_j Z_j prod_{j odd} X_j  ->  spectrum(K) = -spectrum(-K)
print("spectrum(K) = -spectrum(-K)         :", np.allclose(np.sort(np.linalg.eigvalsh(H_chain(L_, 0.4, K=0.3))),
                                                        np.sort(-np.linalg.eigvalsh(H_chain(L_, 0.4, K=-0.3)))))
S_ = Sectors(8, 0.4); sp_ = Spectrum(H_chain(8, 0.4)); tt = np.logspace(-1, 4, 11)
print("sector vs full ED memory, L=8       :", np.abs(S_.memory(S_.Zpm(0), tt) - autocorr(sp_, sp_.op(zdiag(8, 0)), tt)).max())
sp_ = Spectrum(H_chain(10, 0.4)); tt = np.logspace(-1, 3, 9)
print("typicality vs exact OTOC, L=10      :", np.abs(otoc_exact(sp_, 10, 0, range(10), tt) - otoc_typical(sp_, 10, 0, tt, nstates=2)).max())
ch_ = Chain(8, 0.45); tt = np.array([0.5, 3.0, 20.0])
Ue = np.linalg.eigh(H_chain(8, 0.45)); psi = np.eye(256)[:, 7].astype(complex)
ex = Ue[1] @ (np.exp(-1j * Ue[0] * 20.0) * (Ue[1].T @ psi))
print("Chebyshev vs exact propagation, L=8 :", np.abs(ch_.evolve(psi, 20.0) - ex).max())
for g in (0.25, 0.4):
    S_ = Sectors(10, g, 0.0); print(f"K=0, g={g}: q(0.1) = {S_.lowfreq_weight(S_.Zpm(0), [0.1])[0]:.4f}   1-g^2 = {1-g*g:.4f}")
del S_, sp_, ch_

/var/folders/8c/l__lbx_94rb73wg1r1_596t00000gn/T/ipykernel_4726/570183994.py:4: RuntimeWarning: divide by zero encountered in matmul
  Hk = sum(-J * _op(_Z, b, L_) @ _op(_Z, b + 1, L_) for b in range(L_ - 1)) \
/var/folders/8c/l__lbx_94rb73wg1r1_596t00000gn/T/ipykernel_4726/570183994.py:4: RuntimeWarning: overflow encountered in matmul
  Hk = sum(-J * _op(_Z, b, L_) @ _op(_Z, b + 1, L_) for b in range(L_ - 1)) \
/var/folders/8c/l__lbx_94rb73wg1r1_596t00000gn/T/ipykernel_4726/570183994.py:4: RuntimeWarning: invalid value encountered in matmul
  Hk = sum(-J * _op(_Z, b, L_) @ _op(_Z, b + 1, L_) for b in range(L_ - 1)) \
/var/folders/8c/l__lbx_94rb73wg1r1_596t00000gn/T/ipykernel_4726/570183994.py:5: RuntimeWarning: divide by zero encountered in matmul
  + sum(K0 * _op(_Z, b, L_) @ _op(_Z, b + 2, L_) for b in range(L_ - 2)) + sum(-0.4 * _op(_X, i, L_) for i in range(L_))
/var/folders/8c/l__lbx_94rb73wg1r1_596t00000gn/T/ipykernel_4726/570183994.py:5: RuntimeWarning: overflow encountered in 

Hamiltonian builder error           : 4.440892098500626e-16
spectrum(K) = -spectrum(-K)         : True
sector vs full ED memory, L=8       : 1.8200163598436347e-12
typicality vs exact OTOC, L=10      : 0.049144915482433604
Chebyshev vs exact propagation, L=8 : 2.1447212987743313e-14
K=0, g=0.25: q(0.1) = 0.9375   1-g^2 = 0.9375
K=0, g=0.4: q(0.1) = 0.8400   1-g^2 = 0.8400


/var/folders/8c/l__lbx_94rb73wg1r1_596t00000gn/T/ipykernel_4726/191787671.py:45: RuntimeWarning: divide by zero encountered in matmul
  Phi = np.concatenate([psi[:, None], P.T @ (zs * (P @ psi)[:, None])], axis=1)
/var/folders/8c/l__lbx_94rb73wg1r1_596t00000gn/T/ipykernel_4726/191787671.py:45: RuntimeWarning: overflow encountered in matmul
  Phi = np.concatenate([psi[:, None], P.T @ (zs * (P @ psi)[:, None])], axis=1)
/var/folders/8c/l__lbx_94rb73wg1r1_596t00000gn/T/ipykernel_4726/191787671.py:45: RuntimeWarning: invalid value encountered in matmul
  Phi = np.concatenate([psi[:, None], P.T @ (zs * (P @ psi)[:, None])], axis=1)
/var/folders/8c/l__lbx_94rb73wg1r1_596t00000gn/T/ipykernel_4726/191787671.py:48: RuntimeWarning: divide by zero encountered in matmul
  WPhi = np.conj(ph)[:, None] * (M @ (ph[:, None] * Phi))
/var/folders/8c/l__lbx_94rb73wg1r1_596t00000gn/T/ipykernel_4726/191787671.py:48: RuntimeWarning: overflow encountered in matmul
  WPhi = np.conj(ph)[:, None] * (M @ (ph[:, N

## 3. Data generation
### 3a. Level statistics (Fig. 2)
This cell computes the mean gap ratio $\langle r\rangle$ in the sectors resolved by both the parity $P$ and the reflection $R$, with $P=+1$ and $R=\pm1$. It averages over the central 80% of each spectrum and combines the two sectors with weights given by their dimensions.

In [6]:
def rmean(E, frac=0.8):
    E = np.sort(E); n = len(E); lo = int(n * (1 - frac) / 2); E = E[lo:n - lo]
    s = np.diff(E); a, b = s[:-1], s[1:]; return np.mean(np.minimum(a, b) / np.maximum(a, b))

def Hplus_sparse(L, g, K=K0, J=J):
    D2 = 2 ** (L - 1); s = np.arange(D2)
    d = -J * sum(zbits(L, s, b) * zbits(L, s, b + 1) for b in range(L - 1)) + K * sum(zbits(L, s, b) * zbits(L, s, b + 2) for b in range(L - 2))
    rows = [s]; cols = [s]; vals = [d]
    for i in range(1, L):
        rows.append(s ^ (1 << (L - 1 - i))); cols.append(s); vals.append(np.full(D2, -g))
    rows.append(s ^ (D2 - 1)); cols.append(s); vals.append(np.full(D2, -g))
    return sps.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))), shape=(D2, D2))

def refl_perm(L):
    D2 = 2 ** (L - 1); s = np.arange(D2); full = 2 ** L - 1; rev = np.zeros(D2, dtype=np.int64)
    for i in range(L): rev |= ((s >> i) & 1) << (L - 1 - i)
    return np.where(rev < D2, rev, rev ^ full)

def sector_basis(L, rr):
    D2 = 2 ** (L - 1); Rs = refl_perm(L); seen = np.zeros(D2, bool); r_, c_, v_ = [], [], []; col = 0
    for s in range(D2):
        if seen[s]: continue
        t = Rs[s]; seen[s] = seen[t] = True
        if t == s:
            if rr == 1: r_.append(s); c_.append(col); v_.append(1.0); col += 1
        else:
            r_ += [s, t]; c_ += [col, col]; v_ += [1 / np.sqrt(2), rr / np.sqrt(2)]; col += 1
    return sps.csr_matrix((v_, (r_, c_)), shape=(D2, col))

G_LEV = np.array([0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.6, 0.7, 0.8, 1.0, 1.3])
for L in (10, 12, 14):
    if have(f"levels_L{L}") or (L == 14 and not RUN_L14): continue
    t0 = time.time(); Bp, Bm = sector_basis(L, 1), sector_basis(L, -1); r = np.zeros((2, len(G_LEV)))
    for k, g in enumerate(G_LEV):
        H = Hplus_sparse(L, g)
        for a, B in enumerate((Bp, Bm)): r[a, k] = rmean(np.linalg.eigvalsh((B.T @ H @ B).toarray()))
    np.savez(fpath(f"levels_L{L}"), g=G_LEV, r=r, dims=[Bp.shape[1], Bm.shape[1]])
    print(f"levels L={L}: {time.time() - t0:.0f} s")

levels L=10: 0 s
levels L=12: 1 s
levels L=14: 64 s


### 3b. Exact memories and zero-mode weight from the parity sectors (Figs. 3, 6, 7, Table I)
For each $(L,g,K)$ this cell stores the following:
* $A_0(t)=(Z_0|Z_0(t))$, the edge memory;
* $A_{L/2}(t)$, the bulk memory;
* $q(\omega_c)=\frac1D\sum_{|E_m-E_n|<\omega_c}|\langle m|Z_0|n\rangle|^2$, the zero-mode weight.

In [7]:
def run_sector(L, g, K=K0):
    n = f"sec_L{L}_g{g:.2f}_K{K:+.2f}"
    if have(n): return
    t0 = time.time(); S = Sectors(L, g, K); out = dict(t=T_SEC, wc=WC, L=L, g=g, K=K)
    for j in sorted({0, L // 2}):
        M = S.Zpm(j); out[f"A{j}"] = S.memory(M, T_SEC)
        if j == 0: out["q0"] = S.lowfreq_weight(M, WC)
        del M
    np.savez(fpath(n), **out); print(f"{n}: {time.time() - t0:.0f} s")

jobs  = [(L, g, K0) for L in (8, 10, 12) for g in G_TAU]                     # Table I, Fig. 7(b), Fig. 6(c)
jobs += [(12, g, K0) for g in (0.30, 0.70, 1.20)]                            # Fig. 3(a,b)
jobs += [(12, g, 0.0) for g in (0.25, 0.32, 0.40, 0.50)]                     # Fig. 3(c), K = 0 check
jobs += [(L, g, K) for L in (10, 12) for K in (-0.2, -0.4) for g in (0.32, 0.36, 0.40, 0.45, 0.50)]   # Fig. 7(c)
jobs += [(12, g, K0) for g in (0.32, 0.36, 0.40, 0.45, 0.50)]
if RUN_L14: jobs += [(14, g, K0) for g in G_TAU[::-1]]
for L, g, K in jobs: run_sector(L, g, K)

sec_L8_g0.25_K-0.30: 0 s
sec_L8_g0.28_K-0.30: 0 s
sec_L8_g0.32_K-0.30: 0 s
sec_L8_g0.36_K-0.30: 0 s
sec_L8_g0.40_K-0.30: 0 s
sec_L8_g0.45_K-0.30: 0 s
sec_L8_g0.50_K-0.30: 0 s
sec_L10_g0.25_K-0.30: 0 s
sec_L10_g0.28_K-0.30: 0 s
sec_L10_g0.32_K-0.30: 0 s
sec_L10_g0.36_K-0.30: 0 s


/var/folders/8c/l__lbx_94rb73wg1r1_596t00000gn/T/ipykernel_4726/3952802209.py:25: RuntimeWarning: divide by zero encountered in matmul
  return self.Vp.T @ (zbits(self.L, self.s, j)[:, None] * self.Vm)
/var/folders/8c/l__lbx_94rb73wg1r1_596t00000gn/T/ipykernel_4726/3952802209.py:25: RuntimeWarning: overflow encountered in matmul
  return self.Vp.T @ (zbits(self.L, self.s, j)[:, None] * self.Vm)
/var/folders/8c/l__lbx_94rb73wg1r1_596t00000gn/T/ipykernel_4726/3952802209.py:25: RuntimeWarning: invalid value encountered in matmul
  return self.Vp.T @ (zbits(self.L, self.s, j)[:, None] * self.Vm)
/var/folders/8c/l__lbx_94rb73wg1r1_596t00000gn/T/ipykernel_4726/3952802209.py:32: RuntimeWarning: divide by zero encountered in matmul
  out[a:a + chunk] = np.real(np.sum(np.conj(Up) * (W @ Um), axis=0))
/var/folders/8c/l__lbx_94rb73wg1r1_596t00000gn/T/ipykernel_4726/3952802209.py:32: RuntimeWarning: overflow encountered in matmul
  out[a:a + chunk] = np.real(np.sum(np.conj(Up) * (W @ Um), axis=0))

sec_L10_g0.40_K-0.30: 0 s
sec_L10_g0.45_K-0.30: 0 s
sec_L10_g0.50_K-0.30: 0 s
sec_L12_g0.25_K-0.30: 1 s
sec_L12_g0.28_K-0.30: 1 s
sec_L12_g0.32_K-0.30: 1 s
sec_L12_g0.36_K-0.30: 1 s
sec_L12_g0.40_K-0.30: 1 s
sec_L12_g0.45_K-0.30: 1 s
sec_L12_g0.50_K-0.30: 1 s
sec_L12_g0.30_K-0.30: 1 s
sec_L12_g0.70_K-0.30: 1 s
sec_L12_g1.20_K-0.30: 1 s
sec_L12_g0.25_K+0.00: 1 s
sec_L12_g0.32_K+0.00: 1 s
sec_L12_g0.40_K+0.00: 1 s
sec_L12_g0.50_K+0.00: 1 s
sec_L10_g0.32_K-0.20: 0 s
sec_L10_g0.36_K-0.20: 0 s
sec_L10_g0.40_K-0.20: 0 s
sec_L10_g0.45_K-0.20: 0 s
sec_L10_g0.50_K-0.20: 0 s
sec_L10_g0.32_K-0.40: 0 s
sec_L10_g0.36_K-0.40: 0 s
sec_L10_g0.40_K-0.40: 0 s
sec_L10_g0.45_K-0.40: 0 s
sec_L10_g0.50_K-0.40: 0 s
sec_L12_g0.32_K-0.20: 1 s
sec_L12_g0.36_K-0.20: 1 s
sec_L12_g0.40_K-0.20: 1 s
sec_L12_g0.45_K-0.20: 1 s
sec_L12_g0.50_K-0.20: 1 s
sec_L12_g0.32_K-0.40: 1 s
sec_L12_g0.36_K-0.40: 1 s
sec_L12_g0.40_K-0.40: 1 s
sec_L12_g0.45_K-0.40: 1 s
sec_L12_g0.50_K-0.40: 1 s
sec_L14_g0.50_K-0.30: 83 s
sec_L14_g0.

### 3c. OTOC maps at $L=12$ (Figs. 4 and 6)
This cell uses full ED with typicality: two Haar-random states for each source. It computes the edge source $W=Z_0$ at $g=0.3, 0.4, 0.45, 0.5$ and the bulk source $W=Z_5$ at $g=0.3$ and $0.45$.
The typicality noise is about $1/\sqrt{D}$, so curves agree with the paper up to that noise (the random state is drawn in the eigenbasis, whose phase convention depends on LAPACK).

In [8]:
if not have("maps_L12"):
    L = 12; T = np.logspace(-1, 5, 61); out = dict(t=T)
    for g in (0.3, 0.4, 0.45, 0.5):
        t0 = time.time(); sp = Spectrum(H_chain(L, g))
        out[f"edge_{g}"] = otoc_typical(sp, L, 0, T, nstates=2, seed=2)
        if g in (0.3, 0.45): out[f"bulk_{g}"] = otoc_typical(sp, L, L // 2 - 1, T, nstates=2, seed=1)
        print(f"maps g={g}: {time.time() - t0:.0f} s")
    np.savez(fpath("maps_L12"), **out); del sp

/var/folders/8c/l__lbx_94rb73wg1r1_596t00000gn/T/ipykernel_4726/191787671.py:18: RuntimeWarning: divide by zero encountered in matmul
  def op(self, diag): return self.P.T @ (diag[:, None] * self.P)     # diagonal operator in the eigenbasis
/var/folders/8c/l__lbx_94rb73wg1r1_596t00000gn/T/ipykernel_4726/191787671.py:18: RuntimeWarning: overflow encountered in matmul
  def op(self, diag): return self.P.T @ (diag[:, None] * self.P)     # diagonal operator in the eigenbasis
/var/folders/8c/l__lbx_94rb73wg1r1_596t00000gn/T/ipykernel_4726/191787671.py:18: RuntimeWarning: invalid value encountered in matmul
  def op(self, diag): return self.P.T @ (diag[:, None] * self.P)     # diagonal operator in the eigenbasis
/var/folders/8c/l__lbx_94rb73wg1r1_596t00000gn/T/ipykernel_4726/191787671.py:45: RuntimeWarning: divide by zero encountered in matmul
  Phi = np.concatenate([psi[:, None], P.T @ (zs * (P @ psi)[:, None])], axis=1)
/var/folders/8c/l__lbx_94rb73wg1r1_596t00000gn/T/ipykernel_4726/191787

maps g=0.3: 26 s
maps g=0.4: 15 s
maps g=0.45: 25 s
maps g=0.5: 15 s


### 3d. Exact edge-to-bulk OTOC at $L=10$ (Fig. 6c)
This cell computes the exact trace, averaged over the probes $j=4,5,6$. It gives the time $t^*$ at which $\overline C=1/2$.

In [9]:
if not have("leak_L10"):
    L = 10; TL = np.logspace(-1, 7, 81); tl = []
    for g in G_TAU:
        sp = Spectrum(H_chain(L, g)); tl.append(crossing(TL, otoc_exact(sp, L, 0, [4, 5, 6], TL).mean(axis=0), 0.5))
    np.savez(fpath("leak_L10"), g=np.array(G_TAU), tstar=np.array(tl)); print("t* (L=10):", np.round(tl, 1))

/var/folders/8c/l__lbx_94rb73wg1r1_596t00000gn/T/ipykernel_4726/191787671.py:18: RuntimeWarning: divide by zero encountered in matmul
  def op(self, diag): return self.P.T @ (diag[:, None] * self.P)     # diagonal operator in the eigenbasis
/var/folders/8c/l__lbx_94rb73wg1r1_596t00000gn/T/ipykernel_4726/191787671.py:18: RuntimeWarning: overflow encountered in matmul
  def op(self, diag): return self.P.T @ (diag[:, None] * self.P)     # diagonal operator in the eigenbasis
/var/folders/8c/l__lbx_94rb73wg1r1_596t00000gn/T/ipykernel_4726/191787671.py:18: RuntimeWarning: invalid value encountered in matmul
  def op(self, diag): return self.P.T @ (diag[:, None] * self.P)     # diagonal operator in the eigenbasis
/var/folders/8c/l__lbx_94rb73wg1r1_596t00000gn/T/ipykernel_4726/191787671.py:36: RuntimeWarning: divide by zero encountered in matmul
  X = Wt @ V; C[a, k] = 1 - np.real(np.sum(p[:, None] * X * X.T))
/var/folders/8c/l__lbx_94rb73wg1r1_596t00000gn/T/ipykernel_4726/191787671.py:36: Run

t* (L=10): [8393.9 2561.9  896.5  248.1   94.9   35.1    9.5]


### 3e. Operator dependence at $L=12$ (Fig. 5)
The sources are $W\in\{Z_0,X_0,Y_0,Z_1,Z_2,\hat\Psi\}$, where $\hat\Psi=\sum_{|E_m-E_n|<\omega_c}|m\rangle\langle m|Z_0|n\rangle\langle n|/\sqrt q$ is the spectrally filtered zero mode, with $\omega_c=0.1J$.
The OTOC uses the commutator form $C=\tfrac12([W(t),V]|[W(t),V])$. It is averaged over three random states, with probes $Z_j$ and $X_j$. About 1.5 GB of RAM is needed.

In [ ]:
def run_ops(g, L=12, WCUT=0.1):
    n = f"ops_L12_g{g:.2f}"
    if have(n): return
    t0 = time.time(); D = 2 ** L; idx = np.arange(D)
    T = np.logspace(-1, 5, 41); TM = T_SEC
    flip = lambda i: idx ^ (1 << (L - 1 - i))
    def comp_op(name):
        kind, i = name[0], int(name[1:])
        if kind == "Z": return np.diag(zdiag(L, i)).astype(complex)
        M = np.zeros((D, D), complex)
        if kind == "X": M[flip(i), idx] = 1.0
        if kind == "Y": M[flip(i), idx] = 1j * zdiag(L, i)
        return M
    def probes_apply(v):
        return np.stack([zdiag(L, j) * v for j in range(L)] + [v[flip(j)] for j in range(L)], axis=1)
    E, P = np.linalg.eigh(H_chain(L, g)); E = E - E.mean(); om = E[:, None] - E[None, :]
    srcs = {nm: P.T @ comp_op(nm) @ P for nm in ("Z0", "X0", "Y0", "Z1", "Z2")}
    Z0e = srcs["Z0"].real; mask = np.abs(om) < WCUT
    q = np.sum(np.abs(Z0e[mask]) ** 2) / D
    srcs["Psi"] = (Z0e * mask) / np.sqrt(q); del mask
    out = dict(t=T, tm=TM, q=q, wc=WCUT, g=g, L=L)
    U = np.exp(-1j * np.outer(E, TM))
    for nm, W in srcs.items():
        out[f"A_{nm}"] = np.real(np.sum(np.conj(U) * ((np.abs(W) ** 2) @ U), axis=0)) / D
    del U
    prof = []                                     # C_{Psi -> Z_j}(t = 0), exact
    for j in range(L):
        Zj = P.T @ (zdiag(L, j)[:, None] * P); Cm = srcs["Psi"] @ Zj - Zj @ srcs["Psi"]
        prof.append(0.5 * np.sum(np.abs(Cm) ** 2) / D)
    out["Psi_profile_Z"] = np.array(prof)
    rng = np.random.default_rng(7); nst = 3
    for nm, W in srcs.items():
        C = np.zeros((2 * L, len(T)))
        for s in range(nst):
            psi = rng.normal(size=D) + 1j * rng.normal(size=D); psi /= np.linalg.norm(psi)
            Vpsi = P.T @ probes_apply(P @ psi)
            for k, t in enumerate(T):
                ph = np.exp(-1j * E * t)
                Wt = lambda X: np.conj(ph)[:, None] * (W @ (ph[:, None] * X))
                u = Wt(psi[:, None])[:, 0]; Vu = P.T @ probes_apply(P @ u); WVpsi = Wt(Vpsi)
                C[:, k] += 0.5 * np.sum(np.abs(WVpsi - Vu) ** 2, axis=0)
        out[f"C_{nm}"] = C / nst
    np.savez(fpath(n), **out); print(f"{n}: {(time.time() - t0) / 60:.1f} min")

for g in (0.30, 0.40): run_ops(g)

/var/folders/8c/l__lbx_94rb73wg1r1_596t00000gn/T/ipykernel_4726/2722188518.py:17: RuntimeWarning: divide by zero encountered in matmul
  srcs = {nm: P.T @ comp_op(nm) @ P for nm in ("Z0", "X0", "Y0", "Z1", "Z2")}
/var/folders/8c/l__lbx_94rb73wg1r1_596t00000gn/T/ipykernel_4726/2722188518.py:17: RuntimeWarning: overflow encountered in matmul
  srcs = {nm: P.T @ comp_op(nm) @ P for nm in ("Z0", "X0", "Y0", "Z1", "Z2")}
/var/folders/8c/l__lbx_94rb73wg1r1_596t00000gn/T/ipykernel_4726/2722188518.py:17: RuntimeWarning: invalid value encountered in matmul
  srcs = {nm: P.T @ comp_op(nm) @ P for nm in ("Z0", "X0", "Y0", "Z1", "Z2")}
/var/folders/8c/l__lbx_94rb73wg1r1_596t00000gn/T/ipykernel_4726/2722188518.py:24: RuntimeWarning: divide by zero encountered in matmul
  out[f"A_{nm}"] = np.real(np.sum(np.conj(U) * ((np.abs(W) ** 2) @ U), axis=0)) / D
/var/folders/8c/l__lbx_94rb73wg1r1_596t00000gn/T/ipykernel_4726/2722188518.py:24: RuntimeWarning: overflow encountered in matmul
  out[f"A_{nm}"] = n

ops_L12_g0.30: 1.6 min


### 3f. Finite temperature (Fig. 8)
This cell computes the thermal edge memory $A_\beta(t)=\mathrm{Tr}[\rho_\beta Z_0(t)Z_0]$, the thermal zero-mode weight $q_\beta$ and $P(z_1=z_2)$ at $g=0.4$, $K=-0.3$.
The result for $K=+0.3$ follows from the exact map $K\to-K \Leftrightarrow \beta\to-\beta$.

In [ ]:
BET = np.array([-1.0, -0.75, -0.5, -0.25, 0.0, 0.25, 0.5, 0.75, 1.0])
for L in (8, 10, 12, 14):
    n = f"thermal_L{L}"
    if have(n) or (L == 14 and not RUN_L14): continue
    t0 = time.time(); g = 0.4; S = Sectors(L, g, K0); M = S.Zpm(0); W = M ** 2; del M
    z12 = zbits(L, S.s, 1) * zbits(L, S.s, 2)
    dp = np.einsum("sm,s,sm->m", S.Vp, z12, S.Vp); dm = np.einsum("sm,s,sm->m", S.Vm, z12, S.Vm)
    A = np.zeros((len(BET), len(T_SEC))); q = np.zeros(len(BET)); pal = np.zeros(len(BET))
    Emin = min(S.Ep.min(), S.Em.min()); Emax = max(S.Ep.max(), S.Em.max())
    for k, b in enumerate(BET):
        ref = Emin if b > 0 else Emax
        pp = np.exp(-b * (S.Ep - ref)); pm = np.exp(-b * (S.Em - ref)); Z = pp.sum() + pm.sum(); pp /= Z; pm /= Z
        pal[k] = 0.5 * (1 + pp @ dp + pm @ dm)
        for a in range(0, len(T_SEC), 40):
            tt = T_SEC[a:a + 40]
            Up = np.exp(-1j * np.outer(S.Ep - S.E0, tt)); Um = np.exp(-1j * np.outer(S.Em - S.E0, tt))
            A[k, a:a + 40] = np.real(np.sum((pp[:, None] * np.conj(Up)) * (W @ Um), axis=0)
                                     + np.sum(np.conj(Up) * (W @ (pm[:, None] * Um)), axis=0))
        for c in range(0, len(S.Ep), 512):
            om = np.abs(S.Ep[c:c + 512, None] - S.Em[None, :]) < 0.1
            q[k] += np.sum(W[c:c + 512] * om * (pp[c:c + 512, None] + pm[None, :]))
    np.savez(fpath(n), t=T_SEC, beta=BET, A=A, q=q, palign=pal, L=L, g=g); del S, W
    print(f"{n}: {time.time() - t0:.0f} s")

### 3g. Large systems, $L=16$ and $18$ (Figs. 6b, 7a,b, Table I): only with `RUN_LARGE = True`
* The edge memory uses one Haar-random state.
* The OTOC from $Z_0$ uses forward and backward Chebyshev evolution of one random state.

The entries are $(L,g,t_{\max},\text{seed})$, matching the paper's runs.

In [ ]:
MEM_JOBS  = [(16, 0.36, 3000, 31), (16, 0.40, 1000, 11), (16, 0.45, 400, 11), (16, 0.50, 150, 11),
             (18, 0.40, 600, 11), (18, 0.45, 400, 11), (18, 0.50, 150, 11)]
OTOC_JOBS = [(16, 0.45, tgrid(150, 10), [8, 12], 12), (16, 0.50, tgrid(150, 10), [8, 12], 12),
             (18, 0.45, np.array([0.5, 1, 2, 3, 5, 7, 10, 14, 20, 28, 40, 56, 80.0]), [3, 6, 9, 12, 14], 21),
             (18, 0.50, np.array([0.5, 1, 2, 3, 5, 7, 10, 14, 20, 28, 40, 56, 80.0]), [3, 6, 9, 12, 14], 21)]
if RUN_LARGE:
    for L, g, tmax, seed in MEM_JOBS:
        n = f"mem_L{L}_g{g:.2f}"
        if have(n): continue
        t0 = time.time(); ts = tgrid(tmax)
        np.savez(fpath(n), t=ts, edge=cheb_memory(Chain(L, g), 0, ts, seed=seed), L=L, g=g)
        print(f"{n}: {(time.time() - t0) / 60:.1f} min")
    for L, g, ts, probes, seed in OTOC_JOBS:
        n = f"otoc_L{L}_g{g:.2f}"
        if have(n): continue
        t0 = time.time()
        np.savez(fpath(n), t=ts, C=cheb_otoc(Chain(L, g), 0, probes, ts, seed=seed), probes=probes, L=L, g=g)
        print(f"{n}: {(time.time() - t0) / 60:.1f} min")
else:
    print("RUN_LARGE = False: L = 16, 18 skipped (figures use whatever is already in data/)")

## 4. Figures
The figures share one visual language:
* color encodes the field $g$, with the same color for the same $g$ in every figure;
* marker and shade encode the system size $L$;
* solid lines show the edge or unfrustrated case, dashed lines the bulk or frustrated case.

In [ ]:
COL1, COL2 = 3.375, 7.0
plt.rcParams.update({
    "font.family": "serif", "font.serif": ["STIXGeneral", "DejaVu Serif"], "mathtext.fontset": "stix",
    "font.size": 8.5, "axes.labelsize": 9, "axes.titlesize": 8.5, "legend.fontsize": 7.0, "xtick.labelsize": 8,
    "ytick.labelsize": 8, "axes.linewidth": 0.7, "lines.linewidth": 1.2, "lines.markersize": 3.6,
    "xtick.direction": "in", "ytick.direction": "in", "xtick.top": True, "ytick.right": True,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6, "xtick.minor.width": 0.45, "ytick.minor.width": 0.45,
    "xtick.major.size": 3, "ytick.major.size": 3, "xtick.minor.size": 1.6, "ytick.minor.size": 1.6,
    "legend.frameon": False, "legend.handlelength": 1.8, "legend.borderaxespad": 0.3, "legend.labelspacing": 0.25,
    "pdf.fonttype": 42, "ps.fonttype": 42, "savefig.dpi": 600, "figure.dpi": 150,
})
_gmap = plt.get_cmap("viridis")
G_ALL = [0.25, 0.28, 0.3, 0.32, 0.36, 0.4, 0.45, 0.5, 0.6, 0.7, 1.2]
def gcol(g): k = int(np.argmin(np.abs(np.array(G_ALL) - g))); return _gmap(0.03 + 0.87 * k / (len(G_ALL) - 1))
G_MARK = {0.25: "v", 0.3: "o", 0.32: "p", 0.4: "s", 0.45: "D", 0.5: "^"}
L_MARK = {8: "v", 10: "D", 12: "o", 14: "s", 16: "^", 18: "p"}
_lmap = plt.get_cmap("YlOrBr")
def lshade(L): return _lmap(0.32 + 0.66 * (L - 8) / 10)
L_DASH = {8: (0, (1, 1)), 10: (0, (3, 1.2)), 12: (0, (6, 1.5)), 14: (0, (4, 1, 1, 1)), 16: (0, (8, 1.5, 1.5, 1.5)), 18: "-"}
C_EDGE, C_BULK, C_GREY = "#C0392B", "#2C5F8A", "0.45"
OTOC_CMAP = LinearSegmentedColormap.from_list("otoc", ["#FFFFFF", "#C6DBEF", "#4A90C2", "#08306B"])
OP_COL = {"Z0": C_EDGE, "Psi": "#7B241C", "Z1": "#E67E22", "Z2": "#B7950B", "X0": C_BULK, "Y0": "#5DADE2"}
OP_LAB = {"Z0": r"$Z_0$", "Psi": r"$\hat\Psi$", "Z1": r"$Z_1$", "Z2": r"$Z_2$", "X0": r"$X_0$", "Y0": r"$Y_0$"}
OP_LS = {"Z0": "-", "Psi": (0, (5, 1.5)), "Z1": "-", "Z2": "-", "X0": "--", "Y0": (0, (1.5, 1.2))}

def label(ax, s, x=0.03, y=0.95, **kw):
    ax.text(x, y, s, transform=ax.transAxes, ha="left", va="top", fontsize=9, fontweight="bold", **kw)
def logx(ax):
    ax.set_xscale("log"); ax.xaxis.set_major_locator(LogLocator(numticks=12))
    ax.xaxis.set_minor_locator(LogLocator(subs=np.arange(2, 10) * 0.1, numticks=12)); ax.xaxis.set_minor_formatter(NullFormatter())
def save(fig, name):
    fig.savefig(f"{FIGS}/{name}.pdf", bbox_inches="tight", pad_inches=0.02)
    fig.savefig(f"{FIGS}/{name}.png", bbox_inches="tight", pad_inches=0.02, dpi=300); plt.show(); plt.close(fig)
def sec(L, g, K=K0): return load(f"sec_L{L}_g{g:.2f}_K{K:+.2f}")
def best_sec(g):
    for L in (14, 12):
        if have(f"sec_L{L}_g{g:.2f}_K{K0:+.2f}"): return sec(L, g)
def interp_log(t_new, t, y): return np.interp(np.log(t_new), np.log(t), y)
STATS = {}

In [ ]:
# ===== Fig. 2: level statistics =====
fig, ax = plt.subplots(figsize=(COL1, 2.25))
for L, ls in ((10, (0, (2, 1.2))), (12, (0, (5, 1.5))), (14, "-")):
    if not have(f"levels_L{L}"): print(f"Fig. 2: L={L} missing"); continue
    d = load(f"levels_L{L}"); w = np.array(d["dims"], float); r = (w[0] * d["r"][0] + w[1] * d["r"][1]) / w.sum()
    ax.plot(d["g"], r, ls=ls, color="0.2", lw=0.9, zorder=1)
    ax.scatter(d["g"], r, c=[gcol(min(max(g, 0.25), 1.2)) for g in d["g"]], marker=L_MARK[L], s=14 + 3 * (L - 10),
               edgecolors="0.15", linewidths=0.5, zorder=2)
    ax.plot([], [], ls=ls, marker=L_MARK[L], color="0.2", mfc="0.75", mec="0.15", ms=3.4 + 0.25 * (L - 10), lw=0.9, label=fr"$L={L}$")
    STATS[f"r_L{L}"] = dict(zip(np.round(d["g"], 2).tolist(), np.round(r, 3).tolist()))
ax.axhline(0.5307, color=C_GREY, lw=0.8, ls=(0, (4, 2))); ax.axhline(0.3863, color=C_GREY, lw=0.8, ls=(0, (1, 1.5)))
ax.text(0.19, 0.5335, "GOE", va="bottom", fontsize=7.5, color="0.3"); ax.text(0.19, 0.3888, "Poisson", va="bottom", fontsize=7.5, color="0.3")
ax.set_xscale("log"); ax.set_xticks([0.2, 0.3, 0.5, 0.7, 1.0]); ax.set_xticklabels(["0.2", "0.3", "0.5", "0.7", "1"])
ax.xaxis.set_minor_formatter(NullFormatter()); ax.set_xlim(0.18, 1.45); ax.set_ylim(0.37, 0.55)
ax.set_xlabel(r"transverse field $g/J$"); ax.set_ylabel(r"$\langle r\rangle$"); ax.legend(loc="lower right", fontsize=6.6)
save(fig, "fig2_levels")

In [ ]:
# ===== Fig. 3: edge and bulk memory, zero-mode weight q(omega_c) =====
fig, axs = plt.subplots(1, 3, figsize=(COL2, 2.3), gridspec_kw={"wspace": 0.34})
for g in (0.25, 0.3, 0.4, 0.5, 0.7, 1.2):
    d = sec(12, g)
    axs[0].plot(d["t"], d["A0"], "-", color=gcol(g), label=fr"$g={g:g}$")
    axs[1].plot(d["t"], d["A6"], "--", color=gcol(g), label=fr"$g={g:g}$")
for a, nm in ((axs[0], r"$\bf{(a)}$  edge spin $Z_0$"), (axs[1], r"$\bf{(b)}$  bulk spin $Z_6$")):
    logx(a); a.set_ylim(-0.08, 1.04); a.set_xlim(0.1, 1e8); a.set_xlabel(r"time $tJ$")
    a.text(0.97, 0.95, nm, transform=a.transAxes, ha="right", va="top", fontsize=8.5)
axs[0].set_ylabel(r"memory $A(t)$"); axs[0].legend(loc="center right", bbox_to_anchor=(1.0, 0.6), ncol=1, fontsize=6.6)
for g in (0.25, 0.32, 0.4, 0.5):
    d = sec(12, g); axs[2].plot(d["wc"], d["q0"], "-", color=gcol(g), label=fr"$g={g:g}$")
    d0 = sec(12, g, 0.0); axs[2].plot(d0["wc"], d0["q0"], ":", color=gcol(g), lw=1.0)
    axs[2].plot([3.3], [1 - g * g], marker="<", color=gcol(g), ms=4, clip_on=False)      # exact K=0 value 1-g^2
    STATS[f"q_g{g}"] = round(q_of(d), 3); STATS[f"q_K0_g{g}"] = round(q_of(d0), 4)
axs[2].axvspan(0.05, 0.2, color="0.92", lw=0, zorder=0)
logx(axs[2]); axs[2].set_xlim(1e-4, 2.0); axs[2].set_ylim(0, 1.0)
axs[2].set_xlabel(r"frequency window $\omega_c/J$"); axs[2].set_ylabel(r"zero-mode weight $q(\omega_c)$")
axs[2].plot([], [], "k-", lw=1, label=r"$K=-0.3$"); axs[2].plot([], [], "k:", lw=1, label=r"$K=0$")
axs[2].legend(loc="lower right", ncol=2, fontsize=6.4, columnspacing=0.7); label(axs[2], "(c)", x=0.04, y=0.66)
save(fig, "fig3_memory")

In [ ]:
# ===== Fig. 4: OTOC maps (L = 12) =====
d = load("maps_L12"); T = d["t"]; L = 12
fig, axs = plt.subplots(1, 4, figsize=(COL2, 2.35), sharey=True, gridspec_kw={"wspace": 0.08})
panels = [("bulk_0.3", 0.3, 5), ("edge_0.3", 0.3, 0), ("bulk_0.45", 0.45, 5), ("edge_0.45", 0.45, 0)]
for a, (key, g, w), s in zip(axs, panels, "abcd"):
    im = a.pcolormesh(np.arange(L), T, d[key].T, cmap=OTOC_CMAP, vmin=0, vmax=1, shading="nearest", rasterized=True)
    a.set_yscale("log"); a.set_ylim(T[0], T[-1]); a.set_xticks([0, 3, 6, 9, 11])
    a.plot([w], [T[0] * 1.15], marker="^", color=C_EDGE if w == 0 else C_BULK, ms=6, clip_on=False, zorder=5)
    a.set_title(fr"$g={g:g}$, $W=Z_{{{w}}}$ ({'edge' if w == 0 else 'bulk'})", pad=3)
    a.set_xlabel(r"probe site $j$")
    a.text(0.06, 0.965, f"({s})", transform=a.transAxes, va="top", fontsize=9, fontweight="bold",
           bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.85))
axs[0].set_ylabel(r"time $tJ$")
cb = fig.colorbar(im, ax=axs, pad=0.012, fraction=0.025); cb.set_label(r"OTOC $C_{w\to j}(t)$"); cb.outline.set_linewidth(0.5)
save(fig, "fig4_maps")

In [ ]:
# ===== Fig. 5: operator dependence and the zero-mode operator =====
o = load("ops_L12_g0.30"); o4 = load("ops_L12_g0.40"); L = 12; bulk = slice(3, L - 3)
fig, axs = plt.subplots(1, 3, figsize=(COL2, 2.3), gridspec_kw={"wspace": 0.34})
for nm in ("Z0", "Psi", "Z1", "Z2", "X0", "Y0"):
    axs[0].plot(o["tm"], o[f"A_{nm}"], ls=OP_LS[nm], color=OP_COL[nm], label=OP_LAB[nm])
    C = o[f"C_{nm}"][bulk].mean(axis=0)
    axs[1].plot(o["t"], C, ls=OP_LS[nm], color=OP_COL[nm], label=OP_LAB[nm])
    for tq in (10.0, 100.0):
        STATS[f"ops_C{tq:g}_{nm}"] = round(float(interp_log(tq, o["t"], C)), 3)
        STATS[f"ops_A{tq:g}_{nm}"] = round(float(interp_log(tq, o["tm"], o[f"A_{nm}"])), 3)
for a in axs[:2]: logx(a); a.set_xlim(0.1, 1e5); a.set_ylim(-0.04, 1.04); a.set_xlabel(r"time $tJ$")
axs[0].set_ylim(-0.3, 1.04)
axs[0].set_ylabel(r"memory $A_W(t)$"); axs[1].set_ylabel(r"bulk OTOC $\overline{C}_{W\to j}(t)$")
axs[1].legend(loc="upper left", bbox_to_anchor=(-0.01, 1.0), fontsize=6.0, handlelength=1.7, labelspacing=0.15)
axs[1].text(0.97, 0.06, r"$g=0.3$, $L=12$", transform=axs[1].transAxes, ha="right", fontsize=7.5)
j = np.arange(L)
for oo, g in ((o, 0.3), (o4, 0.4)):
    p = oo["Psi_profile_Z"]
    axs[2].semilogy(j, np.maximum(p, 1e-12), "-", marker=G_MARK[g], color=gcol(g), mec="0.15", mew=0.4, ms=3.6, lw=0.9, label=fr"$g={g:g}$")
    k = (j >= 1) & (j <= 6); STATS[f"xi_g{g}"] = round(float(-1 / np.polyfit(j[k], np.log(p[k]), 1)[0]), 2)
axs[2].set_xlabel(r"probe site $j$"); axs[2].set_ylabel(r"$C_{\hat\Psi\to j}(t=0)$"); axs[2].set_xticks([0, 3, 6, 9, 11])
axs[2].legend(loc="upper right"); axs[2].set_ylim(1e-5, 8)
label(axs[0], "(a)", x=0.86, y=0.12); label(axs[1], "(b)", x=0.86, y=0.22); label(axs[2], "(c)", x=0.04, y=0.12)
save(fig, "fig5_operators")

In [ ]:
# ===== Fig. 6: time-doubling law  C(t) = 1 - A(2t) =====
dmap = load("maps_L12"); T = dmap["t"]; L = 12; bulk = slice(3, L - 3)
def A_of(g, tq, L=12):
    if L == 12 and g in (0.3, 0.4): oo = load(f"ops_L12_g{g:.2f}"); return interp_log(tq, oo["tm"], oo["A_Z0"])
    d = sec(L, g) if have(f"sec_L{L}_g{g:.2f}_K{K0:+.2f}") else best_sec(g); return interp_log(tq, d["t"], d["A0"])
def q_g(g): return float(load(f"ops_L12_g{g:.2f}")["q"]) if g in (0.3, 0.4) else q_of(best_sec(g))
fig, axs = plt.subplots(1, 3, figsize=(COL2, 2.35), gridspec_kw={"wspace": 0.36, "width_ratios": [1.15, 1, 1]})
res = {}
for g in (0.3, 0.45):                                                   # (a) time traces
    C = dmap[f"edge_{g}"][bulk].mean(axis=0); A1 = A_of(g, T); A2 = A_of(g, 2 * T)
    axs[0].plot(T, C, G_MARK[g], color=gcol(g), ms=2.8, mew=0, label=fr"$g={g:g}$")
    axs[0].plot(T, 1 - A2, "-", color=gcol(g), lw=1.1)                  # doubling law
    axs[0].plot(T, 1 - A1, ":", color=gcol(g), lw=1.0)                  # naive 1 - A(t)
    axs[0].plot(T, 1 - A1 ** 2 / q_g(g), ls=(0, (5, 1.5, 1, 1.5)), color=gcol(g), lw=0.9)   # 1 - A^2/q
for g in (0.3, 0.4, 0.45, 0.5):                                         # (b) collapse, L = 12
    C = dmap[f"edge_{g}"][bulk].mean(axis=0); A1 = A_of(g, T); A2 = A_of(g, 2 * T); m = T >= 10
    axs[1].plot(1 - A2[m], C[m], G_MARK[g], color=gcol(g), ms=2.8, mew=0, label=fr"$g={g:g}$")
    for nm, pr in (("2t", 1 - A2), ("1t", 1 - A1), ("sq", 1 - A1 ** 2 / q_g(g))):
        r = C[m] - pr[m]; res[f"L12_g{g}_{nm}"] = (round(float(np.abs(r).max()), 3), round(float(np.sqrt(np.mean(r * r))), 3))
for Lb, mk in ((16, "x"), (18, "+")):                                   # (b) collapse, L = 16, 18
    for g in (0.45, 0.5):
        if not have(f"otoc_L{Lb}_g{g:.2f}"): print(f"Fig. 6(b): otoc_L{Lb}_g{g:.2f} missing"); continue
        oo = load(f"otoc_L{Lb}_g{g:.2f}"); pr = list(oo["probes"]); C = oo["C"][pr.index(Lb // 2)]
        A2 = A_of(g, 2 * oo["t"], 14); m = oo["t"] >= 20
        axs[1].plot(1 - A2[m], C[m], mk, color=gcol(g), ms=3.4, mew=0.8)
        r = C[m] - (1 - A2[m]); res[f"L{Lb}_g{g}_2t"] = (round(float(np.abs(r).max()), 3), round(float(np.sqrt(np.mean(r * r))), 3))
axs[1].plot([], [], "x", color="0.3", label=r"$L=16$"); axs[1].plot([], [], "+", color="0.3", label=r"$L=18$")
axs[1].plot([0, 1], [0, 1], color="0.2", lw=0.7, ls=(0, (4, 2)))
logx(axs[0]); axs[0].set_xlim(0.1, 1e5); axs[0].set_ylim(-0.02, 1.02)
axs[0].set_xlabel(r"time $tJ$"); axs[0].set_ylabel(r"bulk OTOC $\overline{C}_{0\to j}(t)$"); axs[0].legend(loc="upper left", bbox_to_anchor=(0.0, 0.88))
axs[1].set_xlim(0, 1); axs[1].set_ylim(0, 1); axs[1].set_aspect("equal", adjustable="box")
axs[1].set_xlabel(r"$1-A_{\rm edge}(2t)$"); axs[1].set_ylabel(r"$\overline{C}_{0\to j}(t)$")
axs[1].legend(loc="lower right", ncol=1, fontsize=6.2, handletextpad=0.3)
dl = load("leak_L10"); g10 = dl["g"]; tc = dl["tstar"]                  # (c) t* versus tau_1/2 / 2, exact L = 10
th = np.array([crossing(sec(10, g)["t"], 1 - sec(10, g)["A0"], 0.5) for g in g10]); ok = np.isfinite(tc)
axs[2].loglog(th[ok] / 2, tc[ok], "o", color="0.2", mfc="white", ms=3.6, label=r"$L=10$, exact")
for gg, a_, b_ in zip(g10[ok], th[ok] / 2, tc[ok]): axs[2].plot([a_], [b_], "o", color=gcol(gg), ms=2.4)
xx = np.array([3, 3e4]); axs[2].loglog(xx, xx, color="0.2", lw=0.7, ls=(0, (4, 2)))
axs[2].set_xlabel(r"$\tau_{1/2}J/2$"); axs[2].set_ylabel(r"$t^{*}J$  ($\overline{C}_{0\to j}=1/2$)"); axs[2].set_xlim(3, 3e4); axs[2].set_ylim(3, 3e4)
axs[2].set_aspect("equal", adjustable="box"); axs[2].legend(loc="upper left", bbox_to_anchor=(0.0, 0.88))
STATS["tstar_over_tau_half"] = dict(zip(np.round(g10[ok], 2).tolist(), np.round(tc[ok] / th[ok], 3).tolist()))
for a, s in zip(axs, "abc"): label(a, f"({s})")
STATS["doubling_residuals_(max,rms)"] = res
save(fig, "fig6_doubling")

In [ ]:
# ===== Fig. 7: edge lifetime tau_edge  (A(tau_edge) = q/2) =====
fig, axs = plt.subplots(1, 3, figsize=(COL2, 2.4), gridspec_kw={"wspace": 0.36, "width_ratios": [1, 1.15, 1]})
gsel = 0.45                                                             # (a) memory versus L
for Lm in (12, 14):
    if have(f"sec_L{Lm}_g{gsel:.2f}_K{K0:+.2f}"):
        d = sec(Lm, gsel); axs[0].plot(d["t"], d["A0"], ls=L_DASH[Lm], color=lshade(Lm), lw=1.3, label=fr"$L={Lm}$ (ED)")
for Lm in (16, 18):
    if have(f"mem_L{Lm}_g{gsel:.2f}"):
        m = load(f"mem_L{Lm}_g{gsel:.2f}"); axs[0].plot(m["t"], m["edge"], ls=L_DASH[Lm], color=lshade(Lm), lw=1.3, label=fr"$L={Lm}$")
qsel = q_of(best_sec(gsel)); axs[0].axhline(qsel / 2, color="0.5", lw=0.6, ls=":"); axs[0].text(0.15, qsel / 2 + 0.02, r"$q/2$", fontsize=7, color="0.35")
logx(axs[0]); axs[0].set_xlim(0.1, 400); axs[0].set_ylim(0, 1.0); axs[0].set_xlabel(r"time $tJ$"); axs[0].set_ylabel(r"$A_{\rm edge}(t)$")
axs[0].legend(loc="lower left", fontsize=6.4); axs[0].text(0.97, 0.95, fr"$g={gsel}$", transform=axs[0].transAxes, ha="right", va="top")
TAU = {}                                                                # (b) tau_edge versus J/g
for f in glob.glob(f"{DATA}/sec_L*_K{K0:+.2f}.npz"):
    d = np.load(f); Lm = int(d["L"]); g = round(float(d["g"]), 2)
    if g in G_TAU: TAU.setdefault(Lm, {})[g] = tau_edge(d["t"], d["A0"], q_of(d))
for f in glob.glob(f"{DATA}/mem_L1[68]_g*.npz"):
    m = np.load(f); Lm = int(m["L"]); g = round(float(m["g"]), 2)
    TAU.setdefault(Lm, {})[g] = tau_edge(m["t"], m["edge"], q_of(best_sec(g)))
for Lm in sorted(TAU):
    gs = np.array(sorted(TAU[Lm])); ts = np.array([TAU[Lm][x] for x in gs]); ok = np.isfinite(ts)
    axs[1].semilogy(1 / gs[ok], ts[ok], "-", marker=L_MARK[Lm], color=lshade(Lm), mfc=lshade(Lm), mec="0.15", mew=0.4, ms=3.6, lw=0.8, label=fr"$L={Lm}$")
gb = np.array([g for g in G_TAU]); tb = np.array([crossing(sec(12, g)["t"], 1 - sec(12, g)["A6"], 0.5) for g in gb])
axs[1].semilogy(1 / gb, tb, "--", marker="o", color=C_BULK, mfc="white", mec=C_BULK, ms=3.2, lw=0.9, label="bulk, $L=12$")
# fit on converged data: the largest L whose tau differs from the next smaller L by less than 15 %
CONV, xs, ys = {}, [], []
for g in sorted({g for Lm in TAU for g in TAU[Lm]}):
    Ls = sorted(Lm for Lm in TAU if g in TAU[Lm] and np.isfinite(TAU[Lm][g]))
    if len(Ls) >= 2 and abs(TAU[Ls[-1]][g] / TAU[Ls[-2]][g] - 1) < 0.15 and Ls[-1] >= 12:
        CONV[g] = (Ls[-1], TAU[Ls[-1]][g]); xs.append(1 / g); ys.append(np.log(TAU[Ls[-1]][g]))
xs, ys = np.array(xs), np.array(ys)
fit, cov = np.polyfit(xs, ys, 1, cov=True)                              # tau ~ exp(c J/g)
fitK, covK = np.polyfit(np.log(xs), ys + np.log(1 / xs), 1, cov=True)   # Kemp-Yao-Laumann: g tau ~ (J/g)^p
STATS["exp_fit_c"] = (round(float(fit[0]), 2), round(float(np.sqrt(cov[0, 0])), 2))
STATS["KYL_fit_p"] = (round(float(fitK[0]), 1), round(float(np.sqrt(covK[0, 0])), 1))
STATS["rms_exp"] = round(float(np.sqrt(np.mean((ys - np.polyval(fit, xs)) ** 2))), 3)
STATS["rms_KYL"] = round(float(np.sqrt(np.mean((ys + np.log(1 / xs) - np.polyval(fitK, np.log(xs))) ** 2))), 3)
STATS["fit_points_(L,tau)"] = {k: (v[0], round(v[1], 1)) for k, v in CONV.items()}
xx = np.linspace(1.9, 4.1, 20)
axs[1].semilogy(xx, np.exp(np.polyval(fit, xx)) * 2.5, color="0.2", lw=0.7)
axs[1].semilogy(xx, xx * np.exp(np.polyval(fitK, np.log(xx))) / 10, color="0.2", lw=0.7, ls=(0, (4, 2)), label=r"$\propto g^{-1}(J/g)^p$")
axs[1].text(2.25, np.exp(np.polyval(fit, 2.25)) * 6.0, fr"$\propto e^{{{fit[0]:.1f}\,J/g}}$", fontsize=7.5, rotation=27, ha="left", va="bottom")
axs[1].set_xlabel(r"$J/g$"); axs[1].set_ylabel(r"edge lifetime $\tau_{\rm edge}J$"); axs[1].set_ylim(0.5, 3e5)
axs[1].legend(loc="lower right", bbox_to_anchor=(1.0, 0.185), ncol=2, fontsize=6.0, columnspacing=0.7)
GK = [0.32, 0.36, 0.40, 0.45, 0.50]; KC = {-0.2: "#1B9E77", -0.3: "#7570B3", -0.4: "#D95F02"}   # (c) K dependence
for K, c in KC.items():
    for Lm, mfc in ((12, None), (10, "white")):
        ts = np.array([tau_edge(sec(Lm, g, K)["t"], sec(Lm, g, K)["A0"], q_of(sec(Lm, g, K))) for g in GK])
        axs[2].semilogy(1 / np.array(GK), ts, "-" if Lm == 12 else ":", marker="o", color=c, mfc=mfc or c, mec=c, ms=3.2, lw=0.9,
                        label=fr"$K={K:+.1f}$" if Lm == 12 else None)
        if Lm == 12: STATS[f"K{K}_slope_L12"] = round(float(np.polyfit(1 / np.array(GK), np.log(ts), 1)[0]), 2)
axs[2].plot([], [], "k-", marker="o", ms=3, lw=0.9, label=r"$L=12$"); axs[2].plot([], [], "k:", marker="o", mfc="white", ms=3, lw=0.9, label=r"$L=10$")
axs[2].set_xlabel(r"$J/g$"); axs[2].set_ylabel(r"$\tau_{\rm edge}J$"); axs[2].legend(loc="upper left", fontsize=6.2, ncol=2, columnspacing=0.6)
axs[2].set_ylim(30, 2e5)
label(axs[0], "(a)", x=0.86, y=0.12); label(axs[1], "(b)", x=0.04, y=0.95); label(axs[2], "(c)", x=0.86, y=0.12)
save(fig, "fig7_tau")

In [ ]:
# ===== Fig. 8: temperature dependence (g = 0.4) =====
fig, axs = plt.subplots(1, 2, figsize=(COL1 * 1.55, 2.2), gridspec_kw={"wspace": 0.42})
TT = {}
for Lm in (8, 10, 12, 14):
    if not have(f"thermal_L{Lm}"): print(f"Fig. 8: L={Lm} missing"); continue
    d = load(f"thermal_L{Lm}"); bet = d["beta"]
    tau = np.array([tau_edge(d["t"], d["A"][k], d["q"][k]) for k in range(len(bet))])
    TT[Lm] = dict(zip(np.round(bet, 2).tolist(), np.round(tau, 1).tolist()))
    pos = bet >= 0; neg = bet <= 0                                      # beta<0 at K=-0.3  <=>  beta>0 at K=+0.3
    axs[0].semilogy(bet[pos], tau[pos], "-", marker=L_MARK[Lm], color=lshade(Lm), mec="0.15", mew=0.4, ms=3.4, lw=0.9, label=fr"$L={Lm}$")
    axs[0].semilogy(-bet[neg], tau[neg], "--", marker=L_MARK[Lm], color=lshade(Lm), mfc="white", mec=lshade(Lm), mew=0.8, ms=3.0, lw=0.9)
    if Lm == max(L_ for L_ in (8, 10, 12, 14) if have(f"thermal_L{L_}")):
        axs[1].plot(bet[pos], d["palign"][pos], "-", marker="o", color="0.2", ms=3, lw=0.9, label=r"$K=-0.3$")
        axs[1].plot(-bet[neg], 1 - d["palign"][neg], "--", marker="s", mfc="white", color="0.45", ms=3, lw=0.9, label=r"$K=+0.3$")
        axs[1].set_ylabel(fr"$P(z_1=z_2)$, $L={Lm}$")
    STATS[f"P(z1=z2)_L{Lm}"] = dict(zip(np.round(bet, 2).tolist(), np.round(d["palign"], 3).tolist()))
axs[0].plot([], [], "k-", lw=0.9, label=r"$K=-0.3$"); axs[0].plot([], [], "k--", lw=0.9, label=r"$K=+0.3$")
axs[0].axvspan(0.375, 1.05, color="0.93", lw=0, zorder=0)
axs[0].set_xlabel(r"inverse temperature $\beta J$"); axs[0].set_ylabel(r"$\tau_{\rm edge}J$"); axs[0].legend(loc="upper left", ncol=2, fontsize=6.0, columnspacing=0.6)
axs[0].set_ylim(5, 1e10); axs[0].set_xlim(-0.05, 1.05)
axs[1].set_xlabel(r"inverse temperature $\beta J$"); axs[1].set_ylim(0, 1); axs[1].set_xlim(-0.05, 1.05); axs[1].legend(loc="center right", fontsize=6.4)
label(axs[0], "(a)", x=0.04, y=0.13); label(axs[1], "(b)", x=0.04, y=0.95)
STATS["thermal_tau_edge"] = TT
save(fig, "fig8_temperature")

## 5. Table I and the numbers quoted in the text

The lifetime fits use, for each $g$, the largest $L$ whose $\tau_{\rm edge}$ has converged (it differs by less than 15% from the next smaller $L$).
The paper values, $c=2.9\pm0.1$ and $p=6.9\pm0.4$, need the $L=14$ data and the $L=16,18$ data. With $L\le12$ only, the fit uses fewer and less converged points and gives different numbers.

In [ ]:
Ls = sorted(TAU)
print("Table I.  Zero-mode weight q(omega_c = 0.1 J) and edge lifetime tau_edge J  (K = -0.3)\n")
print(f"{'g':>5} {'q':>6} " + " ".join(f"{'L=' + str(L):>8}" for L in Ls))
for g in G_TAU:
    row = [f"{TAU[L][g]:8.1f}" if g in TAU[L] and np.isfinite(TAU[L][g]) else f"{'-':>8}" for L in Ls]
    print(f"{g:5.2f} {q_of(best_sec(g)):6.3f} " + " ".join(row))
STATS["Table_I_q"] = {g: round(q_of(best_sec(g)), 3) for g in G_TAU}
STATS["Table_I_tau"] = {L: {g: round(float(v), 1) for g, v in sorted(TAU[L].items())} for L in Ls}

print("\nLifetime fits on converged points:", STATS["fit_points_(L,tau)"])
print(f"  exponential  tau ~ exp(c J/g):      c = {STATS['exp_fit_c'][0]} +- {STATS['exp_fit_c'][1]},  rms(log tau) = {STATS['rms_exp']}")
print(f"  Kemp-Yao-Laumann  g tau ~ (J/g)^p:  p = {STATS['KYL_fit_p'][0]} +- {STATS['KYL_fit_p'][1]},  rms(log tau) = {STATS['rms_KYL']}")
print("  slopes at L = 12 for K = -0.2, -0.3, -0.4:", [STATS[f"K{K}_slope_L12"] for K in (-0.2, -0.3, -0.4)])
print("\nZero-mode weight q(0.1): K=-0.3:", {g: STATS[f"q_g{g}"] for g in (0.25, 0.32, 0.4, 0.5)},
      "\n                        K=0   :", {g: STATS[f"q_K0_g{g}"] for g in (0.25, 0.32, 0.4, 0.5)}, " (exact 1-g^2)")
print("\nDoubling-law residuals (max, rms) of C - prediction, t >= 10 J^-1:")
for k, v in res.items(): print(f"  {k:14s} {v}")
print("  [2t] = 1-A(2t),  [1t] = 1-A(t),  [sq] = 1-A(t)^2/q")
print("\nt*/tau_1/2 (L=10, exact):", STATS["tstar_over_tau_half"])
print("\nOperator dependence at g=0.3, t=100: A_W =", {nm: STATS[f"ops_A100_{nm}"] for nm in OP_LAB},
      "\n                                      C_W =", {nm: STATS[f"ops_C100_{nm}"] for nm in OP_LAB})
print("Decay length of C_{Psi->j}(0):", {g: STATS[f"xi_g{g}"] for g in (0.3, 0.4)})
print("\nThermal tau_edge (g=0.4, K=-0.3; beta<0 equals K=+0.3 at |beta|):")
for L_, v in TT.items(): print(f"  L={L_}: {v}")
json.dump(STATS, open(f"{FIGS}/stats.json", "w"), indent=1, default=lambda x: float(x) if np.isscalar(x) else str(x))
print(f"\nAll numbers saved to {FIGS}/stats.json")